# Meta-analysis with METAL

This notebook runs cross-cohort meta-analysis of summary statistics with **METAL** on the toy `protocol_example` dataset. METAL is a command-line tool that takes a script documenting the input summary-statistic files, the field mapping for each, and the analysis settings. Meta-analysis here is essentially a weighted sum of Z-scores, so the same set of variants must be present across the input cohorts; the input is a list of paths to the per-cohort summary statistics to analyse together.

By default the weighting **SCHEME** is set to standard error (`STDERR`); switching to sample-size weighting would require a sample-size column in the upstream input. The `AVERAGEFREQ`, `MINMAXFREQ`, and `GENOMICCONTROL` options are off by default — enabling any of them requires the corresponding additional column.

The workflow is split into four numbered steps (`METAL_1` … `METAL_4`); the commands below run them in order on the toy input `protocol_example.sumstat_list.tsv` (a tab-separated list with a `#chr` column and one cohort column `protocol_example`). The referenced sumstats carry `chr, pos, A1, A2, beta, se, z, p`, so effect/SE/p-value/alleles are mapped to `beta`/`se`/`p`/`A1` `A2`; with no frequency or sample-size column the default STDERR scheme is used. Results go to `output/metal`, no container.

**Role in the protocol.** METAL is the meta-analysis module: it consumes the per-cohort summary statistics produced by QTL association and combines them. It produces two outputs: (1) a list of meta-analysed summary statistics for downstream MASH and mvSuSiE-RSS analysis, and (2) a list of summary statistics in VCF format that serves as the end product of the analysis.

## Input Files

| File |Used as |
|------|---------|
| `input/gwas/protocol_example.sumstat_list.tsv` | `--sumstat_list_path` (tab-separated list with a `#chr` column and one column per cohort, each pointing to that cohort's summary-statistics file) |
| per-cohort summary-statistics files referenced in the list (e.g. `protocol_example.gwas_sumstats.chr22.tsv`) | read per cohort; must carry `variant_id` plus the effect/SE/p-value/allele columns mapped via `--variant_id` / `--Beta` / `--SE` / `--PVAL` / `--ALLELE` |

The `variant_id` column must be formatted as `chr:pos_ref_alt` (colon after the chromosome, underscores between position, reference and alternate alleles).

## Run the full workflow

Calling the `METAL` target runs all four numbered steps in order (`METAL_1` → `METAL_2` → `METAL_3` → `METAL_4`), since they form a chained numbered pipeline. This single command is the normal entry point; the per-step commands below are provided for running or inspecting each step individually.

**Timing:** Runtime varies by dataset size and compute resources. For the toy chr22 MWE dataset, most steps complete in under 10 minutes on a standard HPC node.

In [ ]:
sos run pipeline/METAL.ipynb METAL \
    --sumstat_list_path input/gwas/protocol_example.sumstat_list.tsv \
    --cwd output/metal \
    -j 2


## Output Files

| File | Description |
| --- | --- |
| `{name}.METAL_script.txt` | the generated METAL analysis script (one per group) |
| `{name}.1.METAL.txt` | raw meta-analysis result table from METAL |
| `{name}.METAL.vcf.bgz` | reformatted meta-analysed summary statistics in VCF layout |
| `{name}.METAL_list.txt` | recipe list mapping each chromosome to its meta-analysed output file |

METAT is a command line tool, takes in a script that documenting the input sumstat files, the field of each of those sumstat input file, and then the end of scipt.

METAL analysis is enssentially a weighted sum of Z score, therefore the input of snps from each chrm can be processed seperately. To make sure the same region was analysis, the input of this workflow shall be a list of path to the sumstat, join by " " to be analyzed at once.

Also, The scheme of weighting at the moment is set to be standard error, changing it to sample size would require the upstream input have a columns named sample size, which they dont have it yet.

At the moment, the AVERAGEFREQ, MINMAXFREQ, and GENOMICCONTROL are by default set to off, turning on each of the option require additional input from upstream file.

The output file name of the seconde step will by default contains a 1 linking the desinateted prefix and surfix, which lead to the f'{wd}/{name}1.METAL.txt' file name

## Command Interface

In [ ]:
sos run pipeline/METAL.ipynb -h

# Pipeline Implementation

The SoS workflow definitions below implement the steps run above: the `[global]` parameter block followed by the four numbered steps `METAL_1`–`METAL_4`.

## Anticipated Results

The pipeline produces output files in the `output/` subdirectory named after the workflow step. Verify success by checking that output files exist and are non-empty. See the **Output** section above for the expected file names and formats.

In [ ]:
[global]
parameter: modular_script_dir = path('code/script')  # override with --modular-script-dir
parameter: sumstat_list_path = path('input/gwas/protocol_example.sumstat_list.tsv')
parameter: variant_id = 'variant_id'
parameter: Sample_size = 'N'
parameter: ALLELE = 'alt ref'
parameter: FREQ   = 'EFFECT_ALLELE_FREQ'
parameter: Beta = 'beta'
parameter: SE = 'se'
parameter: PVAL   = 'pval'
parameter: cwd = path('output/metal')
parameter: SCHEME = 'STDERR'
parameter: AVERAGEFREQ = "OFF"
parameter: MINMAXFREQ  = "OFF"
parameter: GENOMICCONTROL  = "OFF"
parameter: container = ''
import pandas as pd

sumstat_list = pd.read_csv(sumstat_list_path, sep = "\t")
sumstat_inv = sumstat_list.values.tolist()
name = "-".join(sumstat_list.columns.values[1:len(sumstat_list.columns.values)].tolist())

In [ ]:
[METAL_1,generate_METAL_script]
input:  for_each = "sumstat_inv"
output: METAL_script = f'{cwd:a}/{name}.{_sumstat_inv[0]}.METAL_script.txt'
task: trunk_workers = 1, trunk_size = 1, walltime = '2h', mem = '55G', cores = 1, tags = f'{step_name}_{_output[0]:bn}'
bash: expand = "$[ ]", stderr = f'{_output}.stderr', stdout = f'{_output}.stdout', container = container
        echo '
        # Meta-analysis weighted by:
        SCHEME   $[SCHEME]
        
        # Whether do genomics control
        GENOMICCONTROL $[GENOMICCONTROL]
        
        # Whether do AVERAGEFREQ or MINMAXFREQ control
        AVERAGEFREQ $[AVERAGEFREQ]
        MINMAXFREQ $[MINMAXFREQ] 
        OUTFILE $[cwd:a]/$[name].$[_sumstat_inv[0]]. .METAL.txt' > $[_output]

        for i in $[" ".join(_sumstat_inv[1:len(_sumstat_inv)])] ; do 
        echo "     
        MARKER   $[variant_id]
        WEIGHT   $[Sample_size]
        ALLELE   $[ALLELE]
        #FREQ     $[FREQ]
        EFFECT   $[Beta]
        STDERR   $[SE]
        PVAL     $[PVAL]
        PROCESS  $i " >> $[_output];
        done 
        
        echo '
        ANALYZE' >> $[_output]

In [ ]:
[METAL_2,Excute_METAL_script]
input: group_with = "sumstat_inv"
output: f'{cwd}/{name}.{_sumstat_inv[0]}.1.METAL.txt'
task: trunk_workers = 1, trunk_size = 1, walltime = '24h', mem = '55G', cores = 1, tags = f'{step_name}_{_output[0]:bn}'
bash: expand = "$[ ]", stderr = f'{_output[0]}.stderr', stdout = f'{_output[0]}.stdout', container = container
       metal $[_input]

In [ ]:
[METAL_3,Output_reformatting]
input: group_with = "sumstat_inv"
output: f'{cwd:a}/{name}.{_sumstat_inv[0]}.METAL.vcf.bgz',
        sumstat = f'{cwd:a}/{name}.{_sumstat_inv[0]}.METAL.txt'
task: trunk_workers = 1, trunk_size = 1, walltime = '2h', mem = '55G', cores = 1, tags = f'{step_name}_{_output[0]:bn}'
bash: expand = "$[ ]", stderr = f'{_output[0]}.stderr', stdout = f'{_output[0]}.stdout', container = container
    Rscript $[modular_script_dir]/multivariate_genome/metal.R \
        --step reformat \
        --input "$[_input]" \
        --output-sumstat "$[_output[1]]" \
        --output-vcf "$[_output[0]]" \
        --name "$[name]"


In [ ]:
[METAL_4,recipe]
input: output_from("METAL_3")["sumstat"],group_by = "all"
output: f'{cwd}/{name}.METAL_list.txt'
bash: expand = "$[ ]", stderr = f'{_output}.stderr', stdout = f'{_output}.stdout', container = container
    Rscript $[modular_script_dir]/multivariate_genome/metal.R \
        --step recipe \
        --sumstat-list "$[sumstat_list_path]" \
        --name "$[name]" \
        --output "$[_output]" \
        $[_input:r]
